In [1]:
# ============================================
# FEATURE ENGINEERING - MEMBER 2
# Omnichannel Retail Analytics
# ============================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the datasets

retail_df = pd.read_csv("../data/cleaned_retail.csv")
rfm_df = pd.read_csv("../data/customer_rfm.csv")

print("Retail Dataset Shape:", retail_df.shape)
print("RFM Dataset Shape:", rfm_df.shape)

Retail Dataset Shape: (779425, 9)
RFM Dataset Shape: (5878, 4)


In [3]:
# Display first 5 rows

print("Retail Dataset")
display(retail_df.head())

print("\nRFM Dataset")
display(rfm_df.head())

Retail Dataset


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0



RFM Dataset


,Customer ID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


In [4]:
# Check dataset information

print("Retail Dataset Info")
retail_df.info()

print("\nRFM Dataset Info")
rfm_df.info()

Retail Dataset Info
<class 'pandas.DataFrame'>
RangeIndex: 779425 entries, 0 to 779424
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      779425 non-null  int64  
 1   StockCode    779425 non-null  str    
 2   Description  779425 non-null  str    
 3   Quantity     779425 non-null  int64  
 4   InvoiceDate  779425 non-null  str    
 5   Price        779425 non-null  float64
 6   Customer ID  779425 non-null  int64  
 7   Country      779425 non-null  str    
 8   TotalAmount  779425 non-null  float64
dtypes: float64(2), int64(3), str(4)
memory usage: 53.5 MB

RFM Dataset Info
<class 'pandas.DataFrame'>
RangeIndex: 5878 entries, 0 to 5877
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer ID  5878 non-null   int64  
 1   Recency      5878 non-null   int64  
 2   Frequency    5878 non-null   int64  
 3   Monetary     5878 non

In [5]:
# Convert InvoiceDate to datetime

retail_df["InvoiceDate"] = pd.to_datetime(retail_df["InvoiceDate"])

print(retail_df["InvoiceDate"].dtype)

datetime64[us]


In [6]:
# Sort by customer and date
retail_df = retail_df.sort_values(["Customer ID", "InvoiceDate"])

# Calculate time difference between consecutive orders
retail_df["DaysBetweenOrders"] = (
    retail_df.groupby("Customer ID")["InvoiceDate"]
    .diff()
    .dt.days
)

# Average time between orders for each customer
avg_time = (
    retail_df.groupby("Customer ID")["DaysBetweenOrders"]
    .mean()
    .reset_index()
)

avg_time.rename(
    columns={"DaysBetweenOrders": "AvgTimeBetweenOrders"},
    inplace=True,
)

avg_time.head()

,Customer ID,AvgTimeBetweenOrders
0,12346,11.969697
1,12347,1.805430
2,12348,7.240000
3,12349,3.270115
4,12350,0.000000


In [7]:
# Create weekday and weekend purchase ratio

# Day of week (Monday = 0, Sunday = 6)
retail_df["DayOfWeek"] = retail_df["InvoiceDate"].dt.dayofweek

# Weekend = Saturday(5), Sunday(6)
retail_df["IsWeekend"] = retail_df["DayOfWeek"].apply(lambda x: 1 if x >= 5 else 0)

# Count weekday/weekend purchases
weekend_ratio = (
    retail_df.groupby("Customer ID")["IsWeekend"]
    .mean()
    .reset_index()
)

weekend_ratio.rename(
    columns={"IsWeekend": "WeekendPurchaseRatio"},
    inplace=True,
)

weekend_ratio.head()

,Customer ID,WeekendPurchaseRatio
0,12346,0.000000
1,12347,0.180180
2,12348,0.058824
3,12349,0.000000
4,12350,0.000000


In [8]:
# Create Lag Feature (Previous Order Amount)

retail_df = retail_df.sort_values(["Customer ID", "InvoiceDate"])

retail_df["PreviousOrderAmount"] = (
    retail_df.groupby("Customer ID")["TotalAmount"]
    .shift(1)
)

# Average previous order amount for each customer
lag_feature = (
    retail_df.groupby("Customer ID")["PreviousOrderAmount"]
    .mean()
    .reset_index()
)

lag_feature.head()

,Customer ID,PreviousOrderAmount
0,12346,11.298788
1,12347,22.231357
2,12348,39.588000
3,12349,23.728103
4,12350,19.325000


In [9]:
# Merge all engineered features with RFM

final_df = rfm_df.merge(avg_time, on="Customer ID", how="left")
final_df = final_df.merge(weekend_ratio, on="Customer ID", how="left")
final_df = final_df.merge(lag_feature, on="Customer ID", how="left")

final_df.head()

,Customer ID,Recency,Frequency,Monetary,AvgTimeBetweenOrders,WeekendPurchaseRatio,PreviousOrderAmount
0,12346,326,12,77556.46,11.969697,0.000000,11.298788
1,12347,2,8,4921.53,1.805430,0.180180,22.231357
2,12348,75,5,2019.40,7.240000,0.058824,39.588000
3,12349,19,4,4428.69,3.270115,0.000000,23.728103
4,12350,310,1,334.40,0.000000,0.000000,19.325000


In [10]:
# Check missing values

final_df.isnull().sum()

Customer ID               0
Recency                   0
Frequency                 0
Monetary                  0
AvgTimeBetweenOrders    115
WeekendPurchaseRatio      0
PreviousOrderAmount     115
dtype: int64

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

numeric_features = [
    "Recency",
    "Frequency",
    "Monetary",
    "AvgTimeBetweenOrders",
    "WeekendPurchaseRatio",
    "PreviousOrderAmount",
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
    ]
)

print("ColumnTransformer Created Successfully!")

ColumnTransformer Created Successfully!


In [12]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

print("Pipeline Created Successfully!")

Pipeline Created Successfully!


In [13]:
processed_data = pipeline.fit_transform(final_df)

print(processed_data.shape)

(5878, 6)


In [14]:
processed_df = pd.DataFrame(
    processed_data,
    columns=numeric_features
)

processed_df.to_csv(
    "../data/member2_processed.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


In [15]:
final_df.to_csv("../data/feature_engineered_data.csv", index=False)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!
